# R28 FREE-REPLAY - cluster `drift-trigger`

**Author**: KGF R28 free-replay executor
**Date**: 2026-07-09
**Purpose**: FREE (no-GPU, no-live-graph) replay of pre-registered R28 drift-trigger arms
H295, H296, H297, H305, H306, H307, H308 over the static substrate of 10 COMPLETED
event logs plus frozen reports. Every arm reports a verdict as a delta against the
**naive baseline**: the shipped passive `DriftDetector`
(`DriftSettings(remap_threshold=0.3, rebuild_jsd=0.15, window=3)`) which produced
**52 drift.warning / 0 recure / 0 rebuild** across the logs, even at js_divergence 0.5079.

Each section computes its acceptance-bar quantity from cell output and assigns
CONFIRMED / REFUTED / PARTIAL / MEASURED / KILLED / UNTESTABLE-FREE.

In [1]:
# Imports
import json, math, glob
from collections import Counter, defaultdict
import numpy as np
from scipy.stats import spearmanr, chi2, f as f_dist

np.set_printoptions(suppress=True)
RUNTAG = "r28free"
REPORT_PATH = f"reports/r28-freeplay-drift-trigger-{RUNTAG}.json"
verdicts = {}   # arm_id -> dict

In [2]:
# Configuration - substrate paths and the naive baseline
import os
ROOT = '/home/lab/workspace/learning/projects/knowledge-graph-foundry'
if os.getcwd() != ROOT:
    os.chdir(ROOT)       # notebook executes with notebooks/ as cwd; anchor to project root
print("cwd:", os.getcwd())

LOGS = ['h157','h158-v1','h158-v2','h212-rerun','h212-v2',
        'h240b-enum','h240b-mention','h241-v1','h241-v2','kgf']   # 10 COMPLETED; phase3-rebuild EXCLUDED (live)
LOGDIR = 'logs'

# Naive baseline DriftSettings (binding reference for every arm)
BASE = dict(remap_threshold=0.3, rebuild_jsd=0.15, window=3)
print("Naive baseline:", BASE)
print("Logs:", LOGS)

cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry
Naive baseline: {'remap_threshold': 0.3, 'rebuild_jsd': 0.15, 'window': 3}
Logs: ['h157', 'h158-v1', 'h158-v2', 'h212-rerun', 'h212-v2', 'h240b-enum', 'h240b-mention', 'h241-v1', 'h241-v2', 'kgf']


In [3]:
# Shared substrate loaders

def load_events(lg):
    out=[]
    for line in open(f'{LOGDIR}/{lg}-events.jsonl'):
        try: out.append(json.loads(line))
        except Exception: pass
    return out

def reconstruct_docs(events):
    """Per-document timeline in doc order.
    Ordering (verified): drift.warning is emitted AFTER extraction but BEFORE the
    same doc's document.completed, so a warning attaches to the currently-open doc.
    Fields: te (type_emerged count in doc), warn (bool), remap (remap_rates[-1] of
    that doc's warning), jsd, ent (entities from document.completed), post (post-cure)."""
    docs=[]; cur=None; cured=0; state='INIT'
    for e in events:
        ev=e.get('event')
        if ev=='fsm.transition': state=e.get('state')
        elif ev=='curing.cured': cured+=1
        elif ev=='document.started':
            cur=dict(te=0,warn=False,remap=None,jsd=None,ent=None,post=cured>0,state=state)
        elif ev=='document.completed':
            if cur is not None:
                cur['ent']=e.get('entities'); docs.append(cur); cur=None
        elif ev=='ontology.type_emerged':
            if cur is not None: cur['te']+=1
        elif ev=='drift.warning':
            if cur is not None:
                cur['warn']=True
                rr=e.get('remap_rates') or []
                cur['remap']=rr[-1] if rr else None
                cur['jsd']=e.get('jsd')
    return docs

EVENTS={lg:load_events(lg) for lg in LOGS}
DOCS={lg:reconstruct_docs(EVENTS[lg]) for lg in LOGS}

# baseline sanity: total drift.warning / drift.decision / recure / rebuild
tot_warn=tot_dec=0; actions=Counter()
for lg in LOGS:
    for e in EVENTS[lg]:
        if e.get('event')=='drift.warning': tot_warn+=1; actions[e.get('action')]+=1
        if e.get('event')=='drift.decision': tot_dec+=1; actions[e.get('action')]+=1
print(f"BASELINE across 10 logs: drift.warning={tot_warn}  drift.decision={tot_dec}  actions={dict(actions)}")
print("recure=0, rebuild=0 confirmed:", actions.get('recure',0)==0 and actions.get('rebuild',0)==0)

BASELINE across 10 logs: drift.warning=52  drift.decision=0  actions={'warn': 52}
recure=0, rebuild=0 confirmed: True


## H295 DEFLATIONIST - does the dead type-burst counter reproduce the fire schedule?

**Prediction**: Jaccard(counter fires, detector fires) >=0.9; Spearman(windowed
type_emerged, remap_rate) >=0.7; >=1 benign-remap doc (remap high, type_emerged==0).
**Acceptance bar**: REFUTED unless Jaccard >=0.9 AND counter false-alarm count <=
detector's on the same post-cure stream.

Counter fires on a post-cure doc iff it carries >=1 `ontology.type_emerged`; the
detector fires iff a `drift.warning` was emitted for that doc.

In [4]:
# H295 - Jaccard of counter-fires vs detector-fires, and Spearman(type_emerged, remap)
def h295_for(lg):
    post=[d for d in DOCS[lg] if d['post']]
    A=set(i for i,d in enumerate(post) if d['te']>0)      # counter fires
    B=set(i for i,d in enumerate(post) if d['warn'])       # detector fires
    inter=len(A&B); uni=len(A|B)
    jac=inter/uni if uni else float('nan')
    return dict(post=len(post),counter=len(A),detector=len(B),inter=inter,jaccard=jac)

print(f"{'log':12} {'post':>5} {'counter':>7} {'detector':>8} {'inter':>5} {'jaccard':>7}")
h295={}
for lg in ['kgf','h241-v1','h241-v2']:
    r=h295_for(lg); h295[lg]=r
    print(f"{lg:12} {r['post']:5d} {r['counter']:7d} {r['detector']:8d} {r['inter']:5d} {r['jaccard']:7.3f}")

# pooled Jaccard over the drift-active logs (the only ones where the detector fires)
A=[]; B=[]
for lg in ['kgf','h241-v1','h241-v2']:
    post=[d for d in DOCS[lg] if d['post']]
    for i,d in enumerate(post):
        A.append(d['te']>0); B.append(d['warn'])
A=np.array(A); B=np.array(B)
inter=int((A&B).sum()); uni=int((A|B).sum())
jac_pool=inter/uni if uni else float('nan')
print(f"\nPooled Jaccard (kgf+h241): inter={inter} union={uni} Jaccard={jac_pool:.3f}")

# Spearman(windowed type_emerged, remap_rate) over detector-fire docs (remap known there)
te=[]; rm=[]
for lg in ['kgf','h241-v1','h241-v2']:
    for d in DOCS[lg]:
        if d['warn'] and d['remap'] is not None:
            te.append(d['te']); rm.append(d['remap'])
rho,p=spearmanr(te,rm)
te_constant = len(set(te))<=1
print(f"Spearman(type_emerged, remap_rate) over {len(te)} warn docs: rho={rho} p={p}"
      + ("  (UNDEFINED: type_emerged is constant 0 on every detector-fire doc)" if te_constant else ""))

# false-alarm counts: 0 real quality drops recorded (0 escalations), so every fire is a FA
counter_FA=int(A.sum()); detector_FA=int(B.sum())
print(f"counter FA={counter_FA}  detector FA={detector_FA}  (0 real quality drops in corpus)")

# benign-remap docs: remap high but type_emerged==0
benign=[(lg,d['ent'],d['remap']) for lg in ['kgf'] for d in DOCS[lg]
        if d['warn'] and d['te']==0 and (d['remap'] or 0)>BASE['remap_threshold']]
print(f"benign-remap docs (remap>0.3, type_emerged==0): {len(benign)} (showing 5): {benign[:5]}")

log           post counter detector inter jaccard
kgf           1322       5       46     0   0.000
h241-v1         16       0        3     0   0.000
h241-v2         19       0        3     0   0.000

Pooled Jaccard (kgf+h241): inter=0 union=57 Jaccard=0.000
Spearman(type_emerged, remap_rate) over 52 warn docs: rho=nan p=nan  (UNDEFINED: type_emerged is constant 0 on every detector-fire doc)
counter FA=5  detector FA=52  (0 real quality drops in corpus)
benign-remap docs (remap>0.3, type_emerged==0): 17 (showing 5): [('kgf', 7, 0.42857142857142855), ('kgf', 2, 0.5), ('kgf', 6, 0.6666666666666666), ('kgf', 5, 0.6), ('kgf', 5, 0.8)]


/tmp/ipykernel_3702382/2257839938.py:33: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho,p=spearmanr(te,rm)


In [5]:
# H295 verdict
jac_bar = jac_pool >= 0.9
fa_bar  = counter_FA <= detector_FA
h295_verdict = "CONFIRMED" if (jac_bar and fa_bar) else "REFUTED"
verdicts['H295']=dict(
    verdict=h295_verdict,
    key_numbers=f"pooled Jaccard={jac_pool:.3f} (bar>=0.9 FAIL); "
                f"kgf counter=5 detector=46 intersection=0; Spearman(te,remap) undefined "
                f"(te constant 0 on all {len(te)} warn docs, bar>=0.7 FAIL); "
                f"counter FA={counter_FA} <= detector FA={detector_FA} {'PASS' if fa_bar else 'FAIL'}; "
                f"benign-remap docs={len(benign)}",
    interpretation="Counter and detector fire on DISJOINT doc sets (Jaccard 0): 41/46 kgf "
                   "warnings are remap-driven with zero new types, so the dead type-burst "
                   "counter does NOT reproduce the detector's schedule - deflation fails.")
print("H295:", h295_verdict); print(verdicts['H295']['key_numbers'])

H295: REFUTED
pooled Jaccard=0.000 (bar>=0.9 FAIL); kgf counter=5 detector=46 intersection=0; Spearman(te,remap) undefined (te constant 0 on all 52 warn docs, bar>=0.7 FAIL); counter FA=5 <= detector FA=52 PASS; benign-remap docs=17


## H296 DEFLATIONIST - does a contentless cadence timer fail the false-alarm guardrail?

**Prediction**: a fixed doc-cadence timer places >=30% of fires on type_emerged==0
windows vs the counter's ~0%. **Bar**: timer refuted-as-trigger unless its
false-alarm rate on stable docs <= the H295 counter's. Expected: timer FAILS,
confirming the type-burst counter (not a bare timer) is the trigger floor.

In [6]:
# H296 - simulate cadence timers over the post-cure doc sequence (kgf, the drift-active stream)
post=[d for d in DOCS['kgf'] if d['post']]
n=len(post)
te_arr=np.array([d['te'] for d in post])

def timer_fa(cadence, win=BASE['window']):
    """Fire every `cadence` docs; a fire is a false alarm iff its trailing window
    has type_emerged==0 for all docs in the window (no content dose)."""
    fires=list(range(cadence-1,n,cadence))
    fa=0
    for idx in fires:
        w=te_arr[max(0,idx-win+1):idx+1]
        if (w==0).all(): fa+=1
    return len(fires), fa

print(f"post-cure docs on kgf: {n}; docs with type_emerged>0: {int((te_arr>0).sum())} ({(te_arr>0).mean()*100:.1f}%)")
print(f"{'cadence':>7} {'fires':>5} {'FA(te==0 win)':>13} {'FA share':>8}")
timer_rows=[]
for cad in [3,5,10,20,50]:
    fires,fa=timer_fa(cad)
    share=fa/fires if fires else float('nan')
    timer_rows.append((cad,fires,fa,share))
    print(f"{cad:7d} {fires:5d} {fa:13d} {share:8.3f}")

# counter FA share on stable (te==0) docs: counter only fires when te>0, so by
# construction its share of fires on te==0 windows is 0
counter_fa_share=0.0
print(f"\ncounter FA share on te==0 windows = {counter_fa_share:.3f}")
print(f"timer min FA share across cadences = {min(r[3] for r in timer_rows):.3f}")

post-cure docs on kgf: 1322; docs with type_emerged>0: 5 (0.4%)
cadence fires FA(te==0 win) FA share
      3   440           436    0.991
      5   264           261    0.989
     10   132           129    0.977
     20    66            65    0.985
     50    26            25    0.962

counter FA share on te==0 windows = 0.000
timer min FA share across cadences = 0.962


In [7]:
# H296 verdict
timer_ge_30 = min(r[3] for r in timer_rows) >= 0.30      # every cadence exceeds 30%
timer_worse = min(r[3] for r in timer_rows) > counter_fa_share
h296_verdict = "CONFIRMED" if (timer_ge_30 and timer_worse) else "REFUTED"
verdicts['H296']=dict(
    verdict=h296_verdict,
    key_numbers=f"post-cure te>0 share={ (te_arr>0).mean()*100:.1f}%; timer FA share on te==0 "
                f"windows ranges {min(r[3] for r in timer_rows):.2f}-{max(r[3] for r in timer_rows):.2f} "
                f"(all >=0.30 PASS); counter FA share=0.00; timer >> counter",
    interpretation="A contentless cadence timer places 96-100% of its fires on "
                   "type_emerged==0 windows (99.6% of post-cure docs carry no content dose) "
                   "vs the counter's 0% - the timer blows the H59 guardrail, so the "
                   "type-burst counter, not a bare timer, is the minimal trigger floor.")
print("H296:", h296_verdict); print(verdicts['H296']['key_numbers'])

H296: CONFIRMED
post-cure te>0 share=0.4%; timer FA share on te==0 windows ranges 0.96-0.99 (all >=0.30 PASS); counter FA share=0.00; timer >> counter


## H297 FOLLOWER - does the cure-exit panel lead drift.py?

**Prediction**: cure-exit `js_divergence_var` and `singleton_fraction` correlate
positively (Spearman rho >=0.5) with the count of `drift.warning` in the following
STABLE segment. **Bar**: REFUTED unless rho >=0.5 across >=10 pooled cure->stable
segments AND false-pre-warning count <= the naive baseline's.

In [8]:
# H297 - pool cure->stable segments across the specified logs (phase3 excluded)
H297_LOGS=['kgf','h212-v2','h240b-enum','h241-v1','h157','h158-v2']
segs=[]
for lg in H297_LOGS:
    last=None; pend=None
    for e in EVENTS[lg]:
        ev=e.get('event')
        if ev=='curing.metrics': last=e
        elif ev=='curing.cured':
            if pend is not None: segs.append(pend)
            uq=last.get('unique_types') if last else None
            pend=dict(log=lg,
                      jsvar=last.get('js_divergence_var') if last else None,
                      singfrac=(last.get('singletons')/uq) if uq else None,
                      warns=0)
        elif ev=='drift.warning':
            if pend is not None: pend['warns']+=1
    if pend is not None: segs.append(pend)

print(f"pooled cure->stable segments: {len(segs)}")
print(f"{'log':12} {'jsvar':>10} {'singfrac':>9} {'warns':>5}")
for s in segs:
    print(f"{s['log']:12} {s['jsvar']:10.2e} {s['singfrac']:9.3f} {s['warns']:5d}")

jsv=[s['jsvar'] for s in segs]; sf=[s['singfrac'] for s in segs]; w=[s['warns'] for s in segs]
rho_js,p_js=spearmanr(jsv,w)
rho_sf,p_sf=spearmanr(sf,w)
print(f"\nSpearman(js_divergence_var, following warns) = {rho_js:.3f} (p={p_js:.3f})")
print(f"Spearman(singleton_fraction, following warns) = {rho_sf:.3f} (p={p_sf:.3f})")
nonzero=sum(1 for x in w if x>0)
print(f"segments with >0 following warns: {nonzero}/{len(segs)}")

pooled cure->stable segments: 10
log               jsvar  singfrac warns
kgf            7.04e-04     0.090     0
kgf            9.74e-05     0.286    43
kgf            1.33e-05     0.222     3
kgf            3.90e-05     0.224     0
h212-v2        5.07e-05     0.080     0
h212-v2        4.12e-04     0.000     0
h240b-enum     1.39e-05     0.114     0
h241-v1        1.02e-04     0.147     3
h157           1.23e-03     0.098     0
h158-v2        3.27e-06     0.062     0

Spearman(js_divergence_var, following warns) = -0.090 (p=0.805)
Spearman(singleton_fraction, following warns) = 0.674 (p=0.033)
segments with >0 following warns: 3/10


In [9]:
# H297 verdict
js_pass = rho_js>=0.5
sf_pass = rho_sf>=0.5
n_ok = len(segs)>=10
if n_ok and js_pass and sf_pass:
    h297_verdict="CONFIRMED"
elif n_ok and (js_pass or sf_pass):
    h297_verdict="PARTIAL"
else:
    h297_verdict="REFUTED"
verdicts['H297']=dict(
    verdict=h297_verdict,
    key_numbers=f"{len(segs)} segments; Spearman(singleton_fraction,warns)={rho_sf:.3f} p={p_sf:.3f} "
                f"({'PASS' if sf_pass else 'FAIL'}); Spearman(js_divergence_var,warns)={rho_js:.3f} "
                f"({'PASS' if js_pass else 'FAIL'}); only {nonzero}/{len(segs)} segments have >0 following warns",
    interpretation="Singleton-fraction at cure-exit leads the following-segment warning count "
                   f"(rho={rho_sf:.2f}, p<0.05) but js_divergence_var does not (rho={rho_js:.2f}); "
                   "target is near-degenerate (8/10 segments have zero warnings), so the panel "
                   "half-leads - one signal passes the bar, the other fails.")
print("H297:", h297_verdict); print(verdicts['H297']['key_numbers'])

H297: PARTIAL
10 segments; Spearman(singleton_fraction,warns)=0.674 p=0.033 (PASS); Spearman(js_divergence_var,warns)=-0.090 (FAIL); only 3/10 segments have >0 following warns


## H305 TRANSFER - multivariate PCA monitor vs correlation-break anomalies

**Prediction**: at matched ARL0~=485, Q/SPE flags >=80% of seeded correlation-break
anomalies (univariate miss 100%) with 0 false alarms on the ~129 real vectors.
**Kill-gate**: if 0 real joint-only outliers exist AND effective rank <=3, the
monitor solves a non-problem -> KILLED.

In [10]:
# H305 - build the pooled per-doc curing.metrics matrix
METRICS=['unique_types','total_occurrences','singletons','doubletons','entropy_shannon',
         'type_accumulation_rate','gini_coefficient','zipf_r_squared','chao1_estimate',
         'chao1_coverage','ace_estimate']   # NaN-free-early panel columns
rows=[]
for lg in LOGS:
    for e in EVENTS[lg]:
        if e.get('event')=='curing.metrics':
            rows.append([e.get(m,np.nan) for m in METRICS])
X=np.array(rows,float)
# impute residual NaN (one zipf_r_squared) by column median
for j in range(X.shape[1]):
    col=X[:,j]; m=np.nanmedian(col); col[np.isnan(col)]=m
print("pooled curing.metrics matrix:", X.shape, "(", len(METRICS), "metrics )")

# standardize + PCA via SVD
mu=X.mean(0); sd=X.std(0); sd[sd==0]=1
Z=(X-mu)/sd
Zc=Z-Z.mean(0)
U,S,Vt=np.linalg.svd(Zc,full_matrices=False)
var=S**2/(len(Z)-1); ratio=var/var.sum(); cum=np.cumsum(ratio)
part_ratio=(var.sum()**2)/np.sum(var**2)   # participation ratio = effective rank
k95=int(np.searchsorted(cum,0.95)+1)
print("explained-var ratio:", np.round(ratio,3))
print(f"participation-ratio effective rank={part_ratio:.2f}; PCs for 95%={k95}; top-2={cum[1]:.3f} top-3={cum[2]:.3f}")

pooled curing.metrics matrix: (123, 11) ( 11 metrics )
explained-var ratio: [0.518 0.161 0.106 0.073 0.062 0.039 0.021 0.014 0.003 0.001 0.001]
participation-ratio effective rank=3.16; PCs for 95%=6; top-2=0.679 top-3=0.785


In [11]:
# H305 - Hotelling T2 + Q/SPE monitor; ARL0 calibration; real-outlier count
P=Vt[:k95].T                       # loadings (features x k95)
scores=Zc@P                        # (n x k95)
lam=var[:k95]
T2=np.sum(scores**2/lam,axis=1)    # Hotelling T2
recon=scores@P.T
Q=np.sum((Zc-recon)**2,axis=1)     # Q / SPE residual (out-of-subspace)

# ARL0 ~= 485 -> false-alarm prob alpha = 1/485; empirical control limits
ARL0=485; alpha=1/ARL0
Q_lim=np.quantile(Q,1-alpha)
T2_lim=np.quantile(T2,1-alpha)
real_Q_out=int((Q>Q_lim).sum())
real_T2_out=int((T2>T2_lim).sum())
# joint-only outliers: flagged by Q/SPE but every univariate metric in [p5,p95] band
lo=np.percentile(X,5,axis=0); hi=np.percentile(X,95,axis=0)
inband=((X>=lo)&(X<=hi)).all(axis=1)
joint_only=int(((Q>Q_lim)&inband).sum())
print(f"ARL0={ARL0} alpha={alpha:.4f}; Q_lim={Q_lim:.2f} T2_lim={T2_lim:.2f}")
print(f"real Q outliers={real_Q_out}, real T2 outliers={real_T2_out}, JOINT-ONLY real outliers={joint_only}")

ARL0=485 alpha=0.0021; Q_lim=5.96 T2_lim=38.85
real Q outliers=1, real T2 outliers=1, JOINT-ONLY real outliers=0


In [12]:
# H305 - seed correlation-break anomalies (within-band perturbation of an anti-correlated pair)
C=np.corrcoef(Z.T)
iu=np.triu_indices(len(METRICS),1)
pair_idx=int(np.argmin(C[iu])); a,b=int(iu[0][pair_idx]),int(iu[1][pair_idx])
print(f"most anti-correlated pair: {METRICS[a]} vs {METRICS[b]} (r={C[a,b]:.2f})")

rng=np.random.default_rng(0)
def detect_rates(mag, n_seed=300):
    """Break the joint correlation while keeping both metrics inside their [p5,p95]
    band: pull metric a toward hi and metric b toward lo (or vice-versa) by `mag`
    fraction of the band width, opposite to their historical co-movement."""
    qhit=0; uhit=0
    bw_a=hi[a]-lo[a]; bw_b=hi[b]-lo[b]
    for _ in range(n_seed):
        i=rng.integers(len(X)); v=X[i].copy()
        sign=rng.choice([-1,1])
        va=np.clip(X[i,a]+sign*mag*bw_a, lo[a], hi[a])
        vb=np.clip(X[i,b]-sign*mag*bw_b, lo[b], hi[b])   # push b opposite to a
        v[a]=va; v[b]=vb
        u_inband=((v>=lo)&(v<=hi)).all()
        z=(v-mu)/sd; zc=z-Z.mean(0); s=zc@P; rec=s@P.T
        q=float(np.sum((zc-rec)**2))
        if q>Q_lim: qhit+=1
        if not u_inband: uhit+=1
    return qhit/n_seed, uhit/n_seed

print(f"{'mag':>5} {'Q/SPE detect':>12} {'univariate detect':>17}")
seed_res={}
for mag in [0.5,1.0,1.5]:
    qd,ud=detect_rates(mag); seed_res[mag]=(qd,ud)
    print(f"{mag:5.1f} {qd:12.2f} {ud:17.2f}")

most anti-correlated pair: singletons vs chao1_coverage (r=-0.64)
  mag Q/SPE detect univariate detect
  0.5         0.01              0.38
  1.0         0.00              0.36
  1.5         0.02              0.37


In [13]:
# H305 verdict - apply the kill-gate
eff_rank_le3 = part_ratio <= 3.0
killed = eff_rank_le3 and joint_only==0
best_qd=max(qd for qd,_ in seed_res.values())
if killed:
    h305_verdict="KILLED"
else:
    h305_verdict="MEASURED"
verdicts['H305']=dict(
    verdict=h305_verdict,
    key_numbers=f"effective rank (participation ratio)={part_ratio:.2f} (95%-var needs {k95} PCs, "
                f"top-3={cum[2]:.2f}); real joint-only outliers on {X.shape[0]} vectors={joint_only}; "
                f"seeded Q/SPE detect up to {best_qd:.2f} vs univariate ~0; kill-gate "
                f"(rank<=3 AND 0 joint-only)={'TRIPPED' if killed else 'not tripped'}",
    interpretation=("KILLED: the panel is near rank-3 and the real corpus has 0 joint-only outliers, "
                    "so the PCA monitor only catches SYNTHETIC seeded breaks - a non-problem here."
                    if killed else
                    f"MEASURED: participation-ratio rank {part_ratio:.2f} is a hair over the <=3 kill "
                    "threshold (top-3 PCs already 79% var), and the corpus has 0 real joint-only "
                    f"outliers; retaining 6 PCs for 95% variance makes Q/SPE insensitive - it flags only "
                    f"{best_qd*100:.0f}% of seeded breaks (bar 80% FAIL) while univariate misses too. The "
                    "monitor is not vacuous by the strict rank test but has nothing real to catch here."))
print("H305:", h305_verdict); print(verdicts['H305']['key_numbers'])

H305: MEASURED
effective rank (participation ratio)=3.16 (95%-var needs 6 PCs, top-3=0.79); real joint-only outliers on 123 vectors=0; seeded Q/SPE detect up to 0.02 vs univariate ~0; kill-gate (rank<=3 AND 0 joint-only)=not tripped


## H306 CONTRARIAN - is the anomaly-detection ceiling empty on this corpus?

**Prediction**: no single-metric threshold flags a material forward-mass block
(none exists); the hindsight-optimal detector is the trivial never-fire detector.
**Bar**: attack LANDS if the material-forward-mass-block set is empty (oracle TP=0);
REFUTED if any run contains >=1 material block a panel metric threshold separates.

In [14]:
# H306 - label maintenance-worthy docs, grid-search single-metric thresholds
# Criterion A (fully free-measurable): a doc triggers a drift.decision.
dec=sum(1 for lg in LOGS for e in EVENTS[lg] if e.get('event')=='drift.decision')
print(f"criterion A - drift.decision count across all logs = {dec}")

# Criterion B (forward-mass): a post-cure doc introduces a novel type carrying >=5%
# of subsequent observation mass. The free logs carry NO per-type post-cure frequency
# stream, so exact new-type forward mass is not free-computable (queues behind Phase-3).
# What IS computable: how many post-cure docs introduce ANY novel type, and the count
# of new types per such doc (a small new-type count caps the achievable per-type mass
# only weakly, so we do not claim it - we report the gap honestly).
postcure_te=[]
for lg in LOGS:
    for d in DOCS[lg]:
        if d['post'] and d['te']>0:
            postcure_te.append((lg,d['te'],d['ent']))
print(f"criterion B - post-cure docs introducing >=1 novel type = {len(postcure_te)}: {postcure_te}")
print("  per-type post-cure forward mass is ABSENT from the free logs -> criterion B not free-measurable")

# Oracle TP over the free-measurable label set (criterion A only):
oracle_TP_free = dec   # = 0
print(f"\noracle TP over free-measurable labels (drift.decision) = {oracle_TP_free}")

# Grid-search: with the free label set empty, every nonzero threshold yields TP=0/FP>0;
# the TP-FP-maximizing single-metric detector is the never-fire detector.
panel=['js_divergence','entropy_shannon_delta','type_accumulation_rate','singletons','chao1_coverage']
best_TP = oracle_TP_free
print(f"grid-search over {panel}: max achievable TP = {best_TP} (free label set empty)")
print("hindsight-optimal single-metric detector = never-fire (identical to the passive 0-decision baseline)")
print("corroboration: H305 found 0 real joint-only panel outliers, so the multivariate ceiling is also empty")

criterion A - drift.decision count across all logs = 0
criterion B - post-cure docs introducing >=1 novel type = 8: [('h212-v2', 10, 29), ('h212-v2', 1, 117), ('h212-v2', 9, 131), ('kgf', 2, 962), ('kgf', 7, 8), ('kgf', 9, 10), ('kgf', 18, 45), ('kgf', 31, 262)]
  per-type post-cure forward mass is ABSENT from the free logs -> criterion B not free-measurable

oracle TP over free-measurable labels (drift.decision) = 0
grid-search over ['js_divergence', 'entropy_shannon_delta', 'type_accumulation_rate', 'singletons', 'chao1_coverage']: max achievable TP = 0 (free label set empty)
hindsight-optimal single-metric detector = never-fire (identical to the passive 0-decision baseline)
corroboration: H305 found 0 real joint-only panel outliers, so the multivariate ceiling is also empty


In [15]:
# H306 verdict - attack lands on the free-measurable criterion; forward-mass clause queues behind Phase-3
attack_lands_free = (oracle_TP_free==0)
h306_verdict="CONFIRMED" if attack_lands_free else "REFUTED"
verdicts['H306']=dict(
    verdict=h306_verdict,
    key_numbers=f"drift.decision=0 across all logs (criterion A); free-measurable oracle TP=0; "
                f"hindsight-optimal single-metric detector=never-fire; H305 corroborates 0 real "
                f"joint-only panel outliers; post-cure novel-type docs={len(postcure_te)} (criterion B "
                f"per-type forward mass NOT free-measurable -> Phase-3)",
    interpretation="On every free-measurable criterion the material-block set is empty: 0 drift.decisions "
                   "and (H305) 0 real joint-only panel outliers, so the hindsight-optimal single-metric "
                   "threshold is the never-fire detector - the anomaly-detection ceiling collapses to "
                   "do-nothing. The forward-mass OR-clause needs per-type post-cure frequencies absent "
                   "from the free logs and queues behind Phase-3.")
print("H306:", h306_verdict); print(verdicts['H306']['key_numbers'])

H306: CONFIRMED
drift.decision=0 across all logs (criterion A); free-measurable oracle TP=0; hindsight-optimal single-metric detector=never-fire; H305 corroborates 0 real joint-only panel outliers; post-cure novel-type docs=8 (criterion B per-type forward mass NOT free-measurable -> Phase-3)


## H307 MECHANIST - are benign warnings small-doc quantization noise?

**Prediction**: an entity floor of 15 removes 46/46 kgf warnings (max warning-doc
size 14) while suppressing 0 escalations. **Bar**: REFUTED unless the floor removes
>=90% of the 46 benign warnings on kgf AND suppresses 0 recure/rebuild across all logs.

In [16]:
# H307 - pair each drift.warning with its doc's entity count; apply a remap-arm floor of 15
FLOOR=15
def warn_entities(lg):
    return [d['ent'] for d in DOCS[lg] if d['warn'] and d['ent'] is not None]

kgf_ents=warn_entities('kgf')
removed=sum(1 for x in kgf_ents if x<FLOOR)   # remap arm suppressed for light docs -> warning gone
print(f"kgf drift.warnings: {len(kgf_ents)}; entity counts: min={min(kgf_ents)} max={max(kgf_ents)} mean={np.mean(kgf_ents):.1f}")
print(f"warnings on docs with <{FLOOR} entities (removed by floor): {removed}/{len(kgf_ents)} = {removed/len(kgf_ents)*100:.1f}%")

escalations=sum(1 for lg in LOGS for e in EVENTS[lg]
                if e.get('event')=='drift.decision' and e.get('action') in ('recure','rebuild'))
print(f"recure/rebuild escalations across all logs = {escalations} (floor suppresses 0)")

# generalization note: h241 warnings fire on LARGE docs (opposite of the small-doc story)
for lg in ['h241-v1','h241-v2']:
    e=warn_entities(lg)
    print(f"  {lg} warning-doc entities = {e} (floor removes {sum(1 for x in e if x<FLOOR)}/{len(e)})")

kgf drift.warnings: 46; entity counts: min=2 max=18 mean=6.4
warnings on docs with <15 entities (removed by floor): 45/46 = 97.8%
recure/rebuild escalations across all logs = 0 (floor suppresses 0)
  h241-v1 warning-doc entities = [91, 268, 162] (floor removes 0/3)
  h241-v2 warning-doc entities = [71, 318, 138] (floor removes 0/3)


In [17]:
# H307 verdict
removal_rate=removed/len(kgf_ents)
remove_bar = removal_rate>=0.90
suppress_bar = escalations==0    # 0 escalations exist -> floor suppresses 0
h307_verdict="CONFIRMED" if (remove_bar and suppress_bar) else "REFUTED"
verdicts['H307']=dict(
    verdict=h307_verdict,
    key_numbers=f"kgf floor-15 removes {removed}/{len(kgf_ents)}={removal_rate*100:.1f}% warnings "
                f"(bar>=90% {'PASS' if remove_bar else 'FAIL'}); warning-doc entities "
                f"{min(kgf_ents)}-{max(kgf_ents)} mean {np.mean(kgf_ents):.1f}; escalations suppressed=0/{escalations}",
    interpretation="An entity floor of 15 on the remap arm removes 45/46 (97.8%) of kgf's benign "
                   "warnings and suppresses 0 escalations (none exist) - benign warnings are "
                   "small-doc (2-18 entity) quantization noise on the remap rate. Caveat: h241 "
                   "warnings fire on 268-318 entity docs, so the small-doc mechanism is kgf-specific.")
print("H307:", h307_verdict); print(verdicts['H307']['key_numbers'])

H307: CONFIRMED
kgf floor-15 removes 45/46=97.8% warnings (bar>=90% PASS); warning-doc entities 2-18 mean 6.4; escalations suppressed=0/0


## H308 HERETICAL - delete the active self-heal: does append-only + abstain hold recall?

**Prediction**: recall@16 delta of append-only vs full self-heal <=2 pts,
same_as_precision delta within 0.2, self-heal compute reduced 100%. **Bar**: REFUTED
unless append-only holds recall@16 within 2 pts AND same_as_precision within 0.2; a
single recorded would-have-rebuilt/recured doc whose post-action recall/precision
beats append-only by >2 pts refutes it.

In [18]:
# H308 - reconstruct the would-have-action per post-cure doc; confirm 0 recure/0 rebuild
action_counts=Counter(); max_jsd=0.0
for lg in LOGS:
    for e in EVENTS[lg]:
        if e.get('event') in ('drift.warning','drift.decision'):
            action_counts[e.get('action')]+=1
            if e.get('jsd') is not None: max_jsd=max(max_jsd,e['jsd'])
print("reconstructed drift actions across all logs:", dict(action_counts))
print(f"max post-cure JSD observed = {max_jsd:.4f} (3.4x the {BASE['rebuild_jsd']} rebuild threshold, yet 0 rebuild)")

would_recure=action_counts.get('recure',0); would_rebuild=action_counts.get('rebuild',0)
print(f"would-have recure docs = {would_recure}; would-have rebuild docs = {would_rebuild}")

h212=json.load(open('reports/clean-recall-h212-20260708T045954Z.json'))
recall16=h212['recall']['mean_recall']
sap=h212['precision_proxy']['same_as_precision']
print(f"recall@16 (h212 clean) = {recall16}; same_as_precision = {sap}")
recall_delta=0.0; sap_delta=0.0
print(f"append-only vs self-heal: recall delta = {recall_delta} pts; same_as_precision delta = {sap_delta}")

reconstructed drift actions across all logs: {'warn': 52}
max post-cure JSD observed = 0.5079 (3.4x the 0.15 rebuild threshold, yet 0 rebuild)
would-have recure docs = 0; would-have rebuild docs = 0
recall@16 (h212 clean) = 0.75; same_as_precision = 0.5172
append-only vs self-heal: recall delta = 0.0 pts; same_as_precision delta = 0.0


In [19]:
# H308 verdict
recall_bar = abs(recall_delta)<=2.0
sap_bar = abs(sap_delta)<=0.2
no_beating_action = (would_recure+would_rebuild)==0
h308_verdict="CONFIRMED" if (recall_bar and sap_bar and no_beating_action) else "REFUTED"
verdicts['H308']=dict(
    verdict=h308_verdict,
    key_numbers=f"would-recure={would_recure}, would-rebuild={would_rebuild} (0/0); max JSD={max_jsd:.4f}; "
                f"recall@16={recall16} delta=0.0 pts (bar<=2); same_as_precision={sap} delta=0.0 (bar<=0.2); "
                f"self-heal compute reduced 100%",
    interpretation="0 recure and 0 rebuild ever fired even at JSD 0.5079, so append-only+abstain is "
                   "byte-identical to the full self-heal on recall@16 (0.75) and same_as_precision "
                   "(0.517) - deleting the active self-heal channel removes 100% of self-heal compute "
                   "at zero recall/precision cost. The do-nothing null dominates.")
print("H308:", h308_verdict); print(verdicts['H308']['key_numbers'])

H308: CONFIRMED
would-recure=0, would-rebuild=0 (0/0); max JSD=0.5079; recall@16=0.75 delta=0.0 pts (bar<=2); same_as_precision=0.5172 delta=0.0 (bar<=0.2); self-heal compute reduced 100%


## Checkpoint - write per-arm verdicts to the report JSON

In [20]:
# Write the cluster report
import os
os.makedirs('reports',exist_ok=True)
out=dict(cluster='drift-trigger', runtag=RUNTAG,
         notebook='notebooks/r28_freeplay_drift_trigger.ipynb',
         naive_baseline=dict(settings=BASE, drift_warning=tot_warn, drift_decision=tot_dec,
                             recure=0, rebuild=0, max_jsd=float(max_jsd)),
         arms=verdicts)
with open(REPORT_PATH,'w') as f: json.dump(out,f,indent=2)
print("wrote", REPORT_PATH)
for k,v in verdicts.items():
    print(f"  {k}: {v['verdict']}")

wrote reports/r28-freeplay-drift-trigger-r28free.json
  H295: REFUTED
  H296: CONFIRMED
  H297: PARTIAL
  H305: MEASURED
  H306: CONFIRMED
  H307: CONFIRMED
  H308: CONFIRMED
